In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Remove the ID column
df = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

df

In [ ]:
df.info()

- 17 feature columsn (ID and N_days were dropped)
    - 7 categorical (`str`)
    - 12 numerical (10 floats, 2 ints)
- 1 Target columns - `Status`
- 418 records (pretty small dataset, we'll see how it performes)
- Quite a lot of missing values (most are highy correlated with missing drug values)
    - I think that other correlations are mostly linked to the drug value missing


In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.isna().corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation of Missing Values')
plt.show()


NaN in the drug column is correlatd with
- NaN in the Ascites, Hepatomegaly, Spiders categorical columns
- NaN in the Chresterol, Copper, Alk_Phos, SGOT, Tryglicerides numerical columns

It might be necessary to delete records with missing drug values. 
It is even said on the dataset website to do this.

On the other hand there are so little records in this dataset.
Might experiment with creating a new class for Drug (`Unspecified` or something like that) 

## Categorical data analysis

In [ ]:
df_cat = df.select_dtypes(include=['object', 'category', 'str'])
df_cat[df_cat['Drug'].isna()]

df[df['Drug'].isna()]

In [ ]:
df_cat.describe()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(df_cat.columns):
    counts = df_cat[col].value_counts(dropna=False)
    
    x_labels = [str(x) for x in counts.index]
    
    axes[i].bar(x_labels, counts.values, color='skyblue', edgecolor='black')
    axes[i].set_title(f"{col}")
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

- Most of the distributions aren't evened out. If we were to remove the nans the only even distributions are Hepatomegaly and Durg.
- Most patients in this stydy were Female.
- High disproportion in the `Status` column (not good)

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df_cat.isna().corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation of Missing Values in Categorical Columns')
plt.show()

In [ ]:
import math

cols = [c for c in df_cat.columns if c != 'Status']
num_plots = len(cols)

ncols = 4
nrows = math.ceil(num_plots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4 * nrows))
axes = axes.flatten()

for idx, col in enumerate(cols):
    ct = pd.crosstab(df_cat['Status'], df_cat[col], normalize='columns')

    sns.heatmap(ct, annot=True, fmt='.2%', cmap='coolwarm', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'Status vs {col}')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('')


plt.tight_layout()
plt.show()

- Status vs Drug
    - The placebo worked similarly as the D-penicillamine
- Status vs Sex
    - Much more deadly for males
- Status vs Ascites
    - Over 95% who had Ascites died (That's a headline). 
    - This doesn't work the other way around as 35% people who didn't have Ascites also died.
    - But there is a large disproportion in data. There were like 10 times more people without Ascites.  
- Status vs Hepatomegalt
    - Pretty deadly for peaople with Hepatomegaly. For people without mostly censored.
- Status vs Spiders
    - Similar to Hematomegaly
- Status vs Edema
    - People with Edema pretty much dead as hell.
    - The S class is weird because it consists of 2 states. But people with Edema and no therapy/or no edema after the therapy have high risk of death.

## Numerical data analysis

In [ ]:
df_num = df.select_dtypes(include='number')

df_num

In [ ]:
df_num.describe()

- Very large standard deviations in comparisson to mean for Bilirubin, Choresterol, Copper, Alk_Phos. Suggests that the data is spreaded.
- Big differences between values between columns. Standarization/normalization needed.
- A few columns have a lot of missing values. Might need to experiment with inputation techniques.

In [ ]:
n_bins = 20

num_plots = len(df_num.columns)
ncols = 4
nrows = math.ceil(num_plots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(df_num.columns):
    axes[i].hist(df_num[col].dropna(), bins=n_bins, color='skyblue', edgecolor='black')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
n_bins = 60

num_plots = len(df_num.columns)
ncols = 4
nrows = math.ceil(num_plots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(df_num.columns):
    axes[i].hist(df_num[col].dropna(), bins=n_bins, color='skyblue', edgecolor='black')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

- Decent distributions. Most of the columns look at least close to normal distribution.
- To there is like only 1 non normal-esque distribution. Bilirubin looks like exponential distribution.
- N_Days looks weird. Multiple smaller peaks.

In [ ]:
n_bins = 20

num_plots = len(df_num.columns)
ncols = 4
nrows = math.ceil(num_plots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(df_num.columns):
    sns.boxplot(df_num[col], color='skyblue', ax=axes[i])
    axes[i].set_title(col)


plt.tight_layout()
plt.show()

- A lot of outliers for `Bilirubin`, `Choresterol`, `Copper`, `Alk_Phos`, `Prothrombin`.

In [ ]:
plt.figure(figsize=(12, 10))

correlation_matrix = df_num.corr() 

sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation of numeric values')
plt.show()

- The numerical colums aren't correlated too much with each other. The highest absolute corr value is 0.46 (Copper x Bilirubin).
- Each column carry some new information.  

## Correlation between numeric columns and Status

In [ ]:
status_dummies = pd.get_dummies(df['Status'], prefix='Status')
corr_num_status = df_num.join(status_dummies).corr().loc[df_num.columns, status_dummies.columns]

plt.figure(figsize=(8, 6))
sns.heatmap(corr_num_status, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation between numeric features and Status')
plt.show()

- Most of the numeric columns have high positive correlation with the `D` Status. The exceptions are 
    - `N_Days`
    - Albumin
    - Piateles
- The oposite is true for the `C` Status.
- `CL` Status has the lowest correlation with anything. 
    - THe biggest absolute correlation value is with age (-0.22).
    - Might be because of the fact that this group is the smallest.